# Compare three director-color schemes

This notebook compares the original Nematics3D director coloring, the previous sRGB Pareto candidate, and the selected OKLab Pareto knee.

It contains three diagnostics: color spheres, a practical Q-field visualization, and principal-loop color-gradient tubes.

In [ ]:
from pathlib import Path
import sys
import numpy as np

def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'example' / 'data' / 'Q_example_workflow.npy').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')

REPO_ROOT = find_repo_root()
try:
    import nematics3d as n3d
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
    import nematics3d as n3d

from nematics3d.field import n_color_immerse
from nematics3d.classes.q_field_object import QFieldObject
from nematics3d.classes.visual.plot_figure import PlotFigure
from nematics3d.classes.visual.plot_tube import PlotTube, OptsTube
from nematics3d.classes.visual.color import (
    director_color_pareto_034,
    director_color_pareto_oklab_043,
    plot_director_color_sphere,
)
from nematics3d.quick import _auto_quick_Q_visual_params, _resolve_director_spacing_level

DATA_PATH = REPO_ROOT / 'example' / 'data' / 'Q_example_workflow.npy'
Q_data = np.load(DATA_PATH)
Q_data.shape

## Part I — color spheres

In [ ]:
scene_nematics3d = plot_director_color_sphere(n_color_immerse, figure_size=(1000, 1000))
scene_nematics3d

In [ ]:
scene_srgb_pareto = plot_director_color_sphere(director_color_pareto_034, figure_size=(1000, 1000))
scene_srgb_pareto

In [ ]:
scene_oklab_pareto = plot_director_color_sphere(director_color_pareto_oklab_043, figure_size=(1000, 1000))
scene_oklab_pareto

## Part II — practical Q-field comparison

The Q field and all visualization geometry are held fixed; only `n_color` changes.

In [ ]:
grid_normal = (0, 0, 1)
director_spacing = 'medium'
params = _auto_quick_Q_visual_params(Q_data, grid_normal)
director_spacing_config = _resolve_director_spacing_level(director_spacing)
Q_obj = QFieldObject(Q=Q_data, name='director-color-comparison', default_miminum_line_length_smooth=params['smooth_min_line_length'], default_smooth_window_length=params['smooth_window_length'], default_miminum_line_length_visual=params['visual_min_line_length'])
Q_obj.act_lines_smooth(min_line_length=params['smooth_min_line_length'], window_length=params['smooth_window_length'])

In [ ]:
def make_qfield_color_comparison_figure(color_func, label):
    figure = PlotFigure()
    Q_obj.act_visualize_disclination_lines(figure=figure, is_extent=False, min_line_length=params['visual_min_line_length'], line_radius=params['line_radius'])
    Q_obj.calc_bounds.act_visualize(figure=figure, opts=OptsTube(radius=params['extent_radius']), is_reset_camera=False)
    Q_obj.act_visualize_n_plane(figure=figure, is_extent=False, grid_normal=grid_normal, grid_spacing=params['grid_spacing'] * director_spacing_config['grid_spacing_scale'], grid_size=params['grid_size'], grid_origin=params['grid_origin'], n_length=params['n_length'] * director_spacing_config['n_length_scale'], n_radius=params['n_radius'] * director_spacing_config['n_radius_scale'], n_color=color_func, plane_name=f'n-plane-{label}')
    return figure

In [ ]:
figure_q_original = make_qfield_color_comparison_figure(n_color_immerse, 'original')
figure_q_original

In [ ]:
figure_q_srgb = make_qfield_color_comparison_figure(director_color_pareto_034, 'srgb-pareto-034')
figure_q_srgb

In [ ]:
figure_q_oklab = make_qfield_color_comparison_figure(director_color_pareto_oklab_043, 'oklab-pareto-043')
figure_q_oklab

## Part III — principal-loop color-gradient diagnostic

Nine straight `PlotTube` objects show the color evolution along three equal-angular-speed director paths for each color scheme:

- $x\to y\to -x$
- $x\to z\to -x$
- $y\to z\to -y$

Rows are color schemes; columns are director paths. Each tube has identical physical length and is sampled uniformly in director angle $\theta\in[0,\pi]$. Thus spatial position along a tube is directly proportional to director rotation angle.

In [ ]:
theta = np.linspace(0.0, np.pi, 301)
principal_paths = {
    'x -> y -> -x': np.column_stack((np.cos(theta), np.sin(theta), np.zeros_like(theta))),
    'x -> z -> -x': np.column_stack((np.cos(theta), np.zeros_like(theta), np.sin(theta))),
    'y -> z -> -y': np.column_stack((np.zeros_like(theta), np.cos(theta), np.sin(theta))),
}
color_schemes = [
    ('Original Nematics3D', n_color_immerse),
    ('sRGB Pareto 0.34', director_color_pareto_034),
    ('OKLab Pareto 0.43', director_color_pareto_oklab_043),
]

def evaluate_colors(color_func, directors):
    try:
        colors = np.asarray(color_func(directors), dtype=float)
        if colors.shape != directors.shape:
            raise ValueError
    except (ValueError, TypeError):
        colors = np.asarray([color_func(n) for n in directors], dtype=float)
    return np.clip(colors, 0.0, 1.0)

figure_principal_loops = PlotFigure()
tube_length = 10.0
column_spacing = 13.0
row_spacing = 3.0
tube_radius = 0.32

for row, (scheme_name, color_func) in enumerate(color_schemes):
    for col, (path_name, directors) in enumerate(principal_paths.items()):
        x_coord = np.linspace(0.0, tube_length, len(theta)) + col * column_spacing
        y_coord = np.full_like(x_coord, -row * row_spacing)
        coords = np.column_stack((x_coord, y_coord, np.zeros_like(x_coord)))
        PlotTube(
            coords,
            name=f'{scheme_name}: {path_name}',
            figure=figure_principal_loops,
            radius=tube_radius,
            color=evaluate_colors(color_func, directors),
            paint_by='color',
            ambient=1.0,
            diffuse=0.0,
            specular=0.0,
            sides=24,
            is_capping=True,
        )

figure_principal_loops.act_view_xy()
figure_principal_loops

Read the nine tubes row-by-row. Because all rows use the same uniformly parameterized director paths and the same tube geometry, visible differences in gradient uniformity, low-chroma passages, abrupt changes, or nearly constant-color stretches come from the coloring scheme.